## MLlib Regression Experiment with Caching

### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: September 22, 2026

---  


#### INSTRUCTIONS: 

This notebook demonstrates construction of a data pipeline and fitting a linear regression model. It illustrates the runtime difference with and without training data caching.

Carefully review the code below, fill in the missing sections, run the code, and note the results

---

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
import time

# =========================================================
# 0 | SET CONFIGS

LABEL_COLUMN = "median_house_value"
FILENAME_DATA = "california_housing_spark_10000.csv"
MAX_ITER = 20

# =========================================================
# 1 | START SPARK

spark = (
    SparkSession.builder
    .appName("CachingDemo")
    .getOrCreate()
)


# =========================================================
# 2 | READ DATA

#[INSERT CODE TO READ THE DATA FROM THE CSV FILE NAMED ABOVE. IT CAN BE FOUND AT THIS PATH: 
#  /standard/ds7200-apt4c/large_datasets/california_housing_spark_10000.csv]

df = spark.read.csv(FILENAME_DATA, header=True, inferSchema=True)

print("Number of records:", df.count())

df.printSchema()


# =========================================================
# 3 | FEATURES

features = [
    "median_income",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "latitude",
    "longitude"
]


# =========================================================
# 4 | FEATURE ENGINEERING

#[INSERT CODE TO INSTANTIATE A VECTOR ASSEMBLER TAKING FEATURES FROM ABOVE]
# The vector assembler object should be named: assembler

assembler = VectorAssembler(inputCols=features, outputCol="features") # inputCols is the list of source columns; outputCol is the name of the new vector column it creates

#[INSERT CODE TO APPLY STANDARD SCALER TO THE FEATURE COLUMN. SCALE THE DATA BUT DON'T SUBTRACT THE MEAN]
# The scaler object should be named: scaler

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False) 
# inputCol="features": this operates on the vector column that the assembler just made, not the original 8;
# withStd=True: divide by standard deviation (this is the scaling)
# withMean=False: don't subtract mean first. Normally standardization is (x-mean)/std but we are skipping the centering step and just doing x/std

# A pipeline is used to preprocess the data. This eliminates multiple calls to fit(), simplifies the process,
# and it's reusable.

pipeline = Pipeline(
    stages=[
        assembler,
        scaler
    ]
)


# =========================================================
# 5 | CREATE TRAINING DATA

pipeline_model = pipeline.fit(df)

processed = (
    pipeline_model
    .transform(df)
    .select(
        "features",
        "median_house_value"
    )
)

# =========================================================
# 6 | TRAIN / TEST SPLIT

#[INSERT CODE TO RANDOMLY SPLIT THE DATA INTO 80% train / 20% test]

train, test = processed.randomSplit([0.8, 0.2], seed=42)
# randomSplit takes a list of weights (not necessarily just two; could do [.7, .15, .15] for a train/val/test in one call and it hands back that many dfs)

print("Training records:", train.count())
print("Test records:", test.count())


# =========================================================
# 7 | LINEAR REGRESSION

#[INSERT CODE TO INSTANTIATE A LINEAR REGRESSION MODEL.]
# It should use the features column as predictors, LABEL_COLUMN for the target variable

lr = LinearRegression(featuresCol="features", labelCol=LABEL_COLUMN, maxIter=MAX_ITER)
# featuresCol="features" points at single vector column the pipeline built
# labeCol=LABEL_COLUMN this is the target, "median_house_value"
# maxIter=MAX_ITER this is a cap on optimization iterations (Spark's LinearRegression uses iterative solvers rather than a closed form normal equation by default)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/23 16:17:23 WARN Utils: Your hostname, Beans-Legion, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/23 16:17:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/sabine/projects/distributed_computing/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/23 16:17:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 16:17:24 WARN Utils: 

Number of records: 10000
root
 |-- median_house_value: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

Training records: 8079
Test records: 1921


In [2]:
# =========================================================
# 8 | WITHOUT CACHE

print("\n========================================")
print("WITHOUT CACHE")
print("========================================")

start = time.time()

for i in range(5):

    model = lr.fit(train)

    # Force Spark to perform another action (inference) using the training data.
    predictions = model.transform(train)

    rmse = (
        predictions
        .selectExpr(
            "sqrt(avg(pow(median_house_value - prediction, 2))) as rmse"
        )
        .collect()[0]["rmse"]
    )

    print(
        f"Iteration {i + 1}: "
        f"training RMSE = {rmse:.2f}"
    )

no_cache_time = time.time() - start

print("\nTime without cache:", no_cache_time)


WITHOUT CACHE


26/09/23 16:17:29 WARN Instrumentation: [88541a7c] regParam is zero, which might cause numerical instability and overfitting.


Iteration 1: training RMSE = 57768.96


26/09/23 16:17:30 WARN Instrumentation: [965658bb] regParam is zero, which might cause numerical instability and overfitting.


Iteration 2: training RMSE = 57768.96


...

Iteration 3: training RMSE = 57768.96


26/09/23 16:17:30 WARN Instrumentation: [39890a2c] regParam is zero, which might cause numerical instability and overfitting.


Iteration 4: training RMSE = 57768.96


26/09/23 16:17:31 WARN Instrumentation: [72a23726] regParam is zero, which might cause numerical instability and overfitting.


Iteration 5: training RMSE = 57768.96

Time without cache: 2.5901691913604736


In [3]:

# =========================================================
# 9 | WITH CACHE

print("\n========================================")
print("WITH CACHE")
print("========================================")

train_cached = train.cache()

# IMPORTANT:
# Materialize the cache before starting the timer. This is done by prompting an action.
train_cached.count()

start = time.time()

for i in range(5):

    # fit the model using the cached training data
    model = lr.fit(train_cached)

    predictions = model.transform(train_cached)

    rmse = (
        predictions
        .selectExpr(
            "sqrt(avg(pow(median_house_value - prediction, 2))) as rmse"
        )
        .collect()[0]["rmse"]
    )

    print(
        f"Iteration {i + 1}: "
        f"training RMSE = {rmse:.2f}"
    )

cache_time = time.time() - start

print("\nTime with cache:", cache_time)


# =========================================================
# 10 | COMPARISON

print("\n========================================")
print("RESULTS")
print("========================================")

print(f"Without cache: {no_cache_time:.2f} seconds")
print(f"With cache:    {cache_time:.2f} seconds")

if cache_time > 0:
    speedup = no_cache_time / cache_time
    print(f"Speedup:       {speedup:.2f}x")


# =========================================================
# CLEAN UP
# =========================================================

train_cached.unpersist()

spark.stop()


WITH CACHE


26/09/23 16:17:31 WARN Instrumentation: [9b14c933] regParam is zero, which might cause numerical instability and overfitting.


Iteration 1: training RMSE = 57768.96
Iteration 2: training RMSE = 57768.96


26/09/23 16:17:32 WARN Instrumentation: [62e20737] regParam is zero, which might cause numerical instability and overfitting.
26/09/23 16:17:32 WARN Instrumentation: [412e7d36] regParam is zero, which might cause numerical instability and overfitting.


Iteration 3: training RMSE = 57768.96


26/09/23 16:17:32 WARN Instrumentation: [0e01e8b1] regParam is zero, which might cause numerical instability and overfitting.


Iteration 4: training RMSE = 57768.96


26/09/23 16:17:32 WARN Instrumentation: [ca4d8815] regParam is zero, which might cause numerical instability and overfitting.


Iteration 5: training RMSE = 57768.96

Time with cache: 1.0804450511932373

RESULTS
Without cache: 2.59 seconds
With cache:    1.08 seconds
Speedup:       2.40x
